# Story 7: Prototype on a Laptop, then Scale Up
**Who:** A PhD student developing a new workflow in a Jupyter notebook on their laptop.

**What they want:** Iterate quickly on the workflow with small inputs, without waiting in a queue, and later run the
very same code on the cluster with production settings.

**Why plain SLURM is painful:** Debugging a workflow by submitting jobs means waiting in the queue for every typo.
Functions defined in a notebook cannot simply be submitted, they first have to be moved to a script. And once the
workflow works locally, it has to be rewritten for the scheduler.

**How executorlib helps:** The `SingleNodeExecutor` runs on any computer with the same interface as the HPC executors,
including functions defined in the notebook, resources, dependencies and caching. The `TestClusterExecutor` tests the
file-based mode of the `SlurmClusterExecutor` locally. Switching to the cluster changes one line.

This notebook runs on **any computer** - no SLURM is required.

## Step 1: Write the workflow in the notebook
executorlib serializes functions with [cloudpickle](https://github.com/cloudpipe/cloudpickle), so functions defined
directly in the notebook can be submitted - no separate module is required.

In [ ]:
import time


def relax_structure(lattice_constant):
    # Stand-in for an expensive structure relaxation.
    time.sleep(2)
    return lattice_constant * 0.98


def compute_energy(relaxed_lattice_constant):
    # Stand-in for an expensive energy calculation.
    time.sleep(2)
    return (relaxed_lattice_constant - 3.95) ** 2 - 4.0

## Step 2: Iterate quickly with the cache
The `SingleNodeExecutor` uses the cores of the local computer. Futures are passed directly as arguments, just like on
the cluster. With a `cache_directory`, the results of finished calls are stored, so re-running the cell during
development only computes what is new.

In [ ]:
from executorlib import SingleNodeExecutor

cache_directory = "./story4_cache"
lattice_constant_lst = [3.9, 4.0, 4.1]


def run_workflow(exe, lattice_constant_lst):
    relaxed_lst = [exe.submit(relax_structure, a) for a in lattice_constant_lst]
    energy_lst = [exe.submit(compute_energy, relaxed) for relaxed in relaxed_lst]
    return [energy.result() for energy in energy_lst]


start = time.time()
with SingleNodeExecutor(cache_directory=cache_directory) as exe:
    print(run_workflow(exe, lattice_constant_lst))
print(f"first run: {time.time() - start:.1f} seconds")

Now extend the study by one lattice constant. Only the new relaxation and energy calculation are executed, the other
results are loaded from the cache:

In [ ]:
start = time.time()
with SingleNodeExecutor(cache_directory=cache_directory) as exe:
    print(run_workflow(exe, lattice_constant_lst + [4.2]))
print(f"second run: {time.time() - start:.1f} seconds")

Changing the code of a function changes its hash, so all calls of the modified function are executed again - the
cache never returns results of outdated code.

## Step 3: Test the file-based mode of the cluster
The `SingleNodeExecutor` communicates with its workers via sockets, while the `SlurmClusterExecutor` communicates via
files in the `cache_directory` on the shared file system. The `TestClusterExecutor` uses the same file-based
communication as the `SlurmClusterExecutor`, but starts the functions as local processes instead of SLURM jobs. It
accepts the same `resource_dict`, so the call which will later be used on the cluster can be tested locally. If the workflow runs here, the remaining sources of errors on the cluster are the
submission template and the Python environment on the compute nodes.

In [ ]:
from executorlib.api import TestClusterExecutor

test_cache_directory = "./story4_test_cache"

with TestClusterExecutor(
    cache_directory=test_cache_directory,
    resource_dict={"cores": 1, "run_time_max": 600},
) as exe:
    print(run_workflow(exe, lattice_constant_lst))

The cache directory now contains one `_o.h5` file per function call. These are the files the `SlurmClusterExecutor`
exchanges between the login node and the compute nodes:

In [ ]:
import os

sorted(os.listdir(test_cache_directory))

## Step 4: Switch to the cluster
On the login node of the cluster, only the executor changes. The function definitions and the `run_workflow()`
function stay exactly the same; the cluster-specific settings go into the `resource_dict` (see the cluster-specific
settings cell of [Story 1](story-1-submit-and-reconnect.ipynb) for a submission template):

```python
from executorlib import SlurmClusterExecutor

with SlurmClusterExecutor(
    cache_directory="./cache",  # must be on a file system shared with the compute nodes
    resource_dict={"submission_template": submission_template, "cores": 1, "run_time_max": 3600},
) as exe:
    print(run_workflow(exe, lattice_constant_lst))
```

Inside an allocation, for many short calculations, use the `SlurmJobExecutor` instead - like the `SingleNodeExecutor`
it communicates via sockets ([Story 2](story-2-parameter-study.ipynb)).

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)
shutil.rmtree(test_cache_directory, ignore_errors=True)

## What you learned
* The `SingleNodeExecutor` runs the same workflow locally, including functions defined in a notebook.
* A `cache_directory` skips finished calculations while iterating on a workflow.
* The `TestClusterExecutor` from `executorlib.api` tests the file-based communication of the `SlurmClusterExecutor`
  without a cluster.
* Moving to the cluster means replacing the executor class and adding the cluster-specific `resource_dict`.

**Reference:** [Single Node Executor](https://executorlib.readthedocs.io/en/latest/1-single-node.html) (including the
[TestClusterExecutor](https://executorlib.readthedocs.io/en/latest/1-single-node.html#testclusterexecutor) and the [cache](https://executorlib.readthedocs.io/en/latest/1-single-node.html#cache)),
[HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html) and [HPC Job Executor](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html).